# TME 310 coding assignment 05

## Instructions

1. Solve each problem in the code cell below it. Add cells if you need them.
2. Some problems end with a task that begins "In a markdown cell". Answer it in the cell headed **Your explanation** under the code, replacing the placeholder text. A comment inside a code cell does not count.
3. If a problem takes more than 30 minutes, stop and get help: the [course AI tutor](https://uwt-tme-310.github.io/TME_310_A/syllabus/polya_ai.html), office hours, or your group.
4. When you finish, restart the kernel, run all cells, export the notebook as HTML or PDF, and upload it on Canvas.

## Learning objectives

- Create, validate, and use models of physical systems
- Use code-based numerical tools

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Problem 1

Imagine a hot sphere of steel with an initial temperature of 1200 K sitting in a very large room with an ambient temperature of 300 K. If heat is lost only by radiation, the Stefan-Boltzmann law gives the rate of change of the sphere's temperature:

$\displaystyle \frac{dT}{dt} = -k \left(T^4 - T_{\infty}^4\right)$

where $T_{\infty}$ is the ambient temperature and $k$ is a constant that combines the emissivity, surface area and the Stefan-Boltzmann constant. For this sphere, $k = 2.207 \times 10^{-12}$ K⁻³/s.

There is no closed-form solution for $T(t)$.

---

1. Write a function `dTdt(t, T)` for the derivative, and a function `euler(dydt, y0, t_end, dt)` that returns arrays of time and the Euler-method solution (or reuse the one from class).
2. Estimate the temperature after 1000 s with step sizes of 100 s, 50 s and 10 s. Print the three estimates and the difference between the 100 s and 50 s results.
3. Plot the three solutions against time on one set of axes with a legend.
4. In a markdown cell: the three curves do not lie on top of each other. Which one is closest to the true solution, how do you know without a closed form, and in which direction (too hot or too cold) is the error of the larger steps?

In [ ]:
# Your code here

### Your explanation

Replace this text with your answer to the last task of the problem (the one that begins "In a markdown cell"). Write it here, in this markdown cell, not as a comment in a code cell.

## Problem 2

The [pendulum project](https://uwt-tme-310.github.io/TME_310_A/extra_labor/proj_phyphox.html) uses recordings from the phyphox app on a phone swinging at the end of a string. `pendulum_data.csv` is one such export, from the "Acceleration (without g)" experiment: a header row, then one row per sample with the time in seconds, the acceleration along the phone's x, y and z axes in m/s², and the magnitude. The phone was released from rest at a small angle for a single trial of 30 s. The loading step is done for you in the cell below.

Use what you know about pendulums to find the period of this one and the length of its arm.

---

1. Plot the three components against time on one set of axes with a legend, and inspect them. For your consideration: which component is a suitable signal for identifying the period, and how do you think the phone was oriented as it swung?
2. Find the zero crossings of the signal you chose (`crossings = np.where(np.diff(np.sign(a)) != 0)[0]` gives the index of the sample just before each sign change in `a`) and plot the signal for the whole 30 s with the crossings marked.
3. Print the period, and the length of the pendulum arm you think it implies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt("pendulum_data.csv", delimiter=",", skiprows=1)   # skip the quoted header row
t = data[:, 0]        # time (s)
a_x = data[:, 1]      # acceleration along the phone's x axis (m/s^2)
a_y = data[:, 2]
a_z = data[:, 3]
print(data.shape)     # (rows, columns)

In [ ]:
# Your code here

## Problem 3

Every slope analysis so far has used one friction angle, $\phi = 30°$, as if it were known exactly. It is not: $\phi$ comes from a handful of laboratory tests, and the results scatter. One way to carry that uncertainty through the model is to draw many values of $\phi$ at random from a distribution that describes the scatter, run the model once for each, and look at the mean of the answers.

Same uniform slope as before, $H = 10$ m, $\beta = 60°$, $\gamma = 19$ kN/m³, $c = 15$ kPa and a failure plane at $\theta = 45°$, with the closed form from the previous assignment:

$$FS = \frac{cL + W\cos\theta\tan\phi}{W\sin\theta}, \qquad L = \frac{H}{\sin\theta}, \qquad W = \gamma\,\frac{H^2}{2}\left(\cot\theta - \cot\beta\right)$$

Take $\phi$ to be normally distributed with a mean of 30° and a standard deviation of 5°. numpy's random generator draws a value from that distribution each time it is called:

```python
rng = np.random.default_rng(310)       # the seed makes your draws repeatable
phi_deg = rng.normal(30, 5)            # one draw
```

**When have you run enough?** The mean of the $FS$ values you have so far is an approximation of the true mean, and each new draw changes it a little, by less and less as the number of draws $N$ grows. That is the same picture as the converging estimates in the lesson on approximation and error, and the stopping rule is the same kind of rule: stop once the mean has stopped changing by more than a tolerance. Statistics says how big the remaining error is. If $s$ is the standard deviation of the $N$ values of $FS$, the true mean lies within about $2s/\sqrt{N}$ of your mean 95% of the time; that range is the **95% confidence interval**. The stopping criterion for this problem is that $2s/\sqrt{N}$ is no more than 1% of the mean, so that the interval spans 2% in all.


---

Run as many simulations as it takes to meet the stopping criterion, then print how many that was, the mean $FS$ and $2s/\sqrt{N}$. Plot the $FS$ from each run against the run number, with a horizontal line at the mean.

How you get there is up to you. If you want a starting point, here is one approach that uses only tools you have already seen. It is not a set of instructions, and you will not be marked on whether you followed it:

- preallocate a large array of `np.nan` to hold the results, say `fs = np.full(5000, np.nan)`;
- in a loop, draw $\phi$, calculate $FS$ and store it in the next empty slot, then recalculate the mean and standard deviation of the results so far (`np.nanmean` and `np.nanstd` ignore the empty slots);
- stop once the stopping criterion is met, but not before you have at least 20 results, since the standard deviation of a handful of values means little.

In [ ]:
# Your code here

## Extra problem (optional)

*This problem is an [extra labor](https://uwt-tme-310.github.io/TME_310_A/extra_labor/). It is not required for the assignment to be complete. Completing it adds 0.1 to your grade (extra problems count up to 0.3 in total), and you can submit it any time before the end of the quarter. Expect it to take about two hours.*

The simple pendulum model $T = 2\pi\sqrt{L/g}$ comes from the small-angle approximation $\sin\theta \approx \theta$. The full equation of motion for a pendulum released from rest at angle $\theta_0$ is

$$\frac{d^2\theta}{dt^2} = -\frac{g}{L}\sin\theta$$

a second-order equation, which Euler's method handles as two first-order equations: $d\theta/dt = \omega$ and $d\omega/dt = -(g/L)\sin\theta$.

---

1. Write `simulate_pendulum(theta0_deg, L, dt, t_end, g=9.81)` that steps both equations with Euler's method and returns arrays of time and angle. Use a *semi-implicit* step: update $\omega$ first, then use the **new** $\omega$ to update $\theta$. (Plain Euler, which uses the old $\omega$, makes the swing grow every cycle; try it once and see.)
2. For $L = 1$ m and $\Delta t = 0.001$ s, simulate 10 s for release angles of 5°, 30°, 60°, 90° and 120°, and plot $\theta(t)$ for all five on one set of axes.
3. Measure the period of each simulation from the times at which $\theta$ crosses zero (a sign change between successive samples; `np.sign` and `np.diff`, or a mask, will find them). Print $T$ for each release angle and the ratio $T / T_0$ with $T_0 = 2\pi\sqrt{L/g}$.
4. Plot $T/T_0$ against $\theta_0$. In a markdown cell: at what release angle is the simple model 1% off? 10%? How does the step size affect your answer (try $\Delta t = 0.01$ s)?

In [ ]:
# Your code here

### Your explanation

Replace this text with your answer to the last task of the problem (the one that begins "In a markdown cell"). Write it here, in this markdown cell, not as a comment in a code cell.